# Fine-tuning + éval compost — version LOCALE (GPU)

Équivalent local de `colab_finetune.ipynb`, sans Colab ni Drive : tout tourne sur ton GPU et s'enregistre dans `runs/`.

**Prérequis** : venv installé, et le `best.pt` **pré-entraîné** récupéré sur ta machine.

**Utilisation** : règle les 2 variables dans la cellule *Config*, puis *Run All* (Kernel → Restart & Run All). Les étapes s'enchaînent toutes seules.

In [1]:
# --- Config : les 2 seules choses à régler ---
PRETRAIN = "/home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/runs/train_29-06_17h13/weights/best.pt"   # modèle PRE-ENTRAINE (a changer selon le run)
DEVICE   = "0"          # "0" = GPU, "cpu" si pas de GPU
EPOCHS   = 50
BATCH    = 8            # baisse à 4 si erreur mémoire GPU

import os, glob, subprocess
os.chdir(os.path.expanduser("~/stage/Compost_Waste_Yolo/compost-yolo"))
PYTHON = os.path.join(os.getcwd(), "venv", "bin", "python")   # python du venv (contient compost_detection)
# le kernel Jupyter exporte MPLBACKEND=...inline, invalide hors notebook -> backend fichier
ENV = {**os.environ, "MPLBACKEND": "Agg"}
def run(cmd):
    if cmd.startswith("python "):
        cmd = PYTHON + cmd[6:]          # force le python du venv, quel que soit le kernel Jupyter
    print("\n$", cmd)
    if subprocess.call(cmd, shell=True, env=ENV) != 0:
        raise RuntimeError("échec de : " + cmd)
print("dossier :", os.getcwd(), "| python :", PYTHON)

dossier : /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo | python : /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/venv/bin/python


## 1. Split — met de côté le test compost + prépare le train/val du fine-tuning

In [2]:
run("python scripts/split_captures.py --source data/raw/captures --output data/finetune")
run("python scripts/prepare_dataset.py --source data/finetune/captures_finetune "
    "--output data/finetune/dataset_finetune --ratios 0.85 0.15 0")


$ /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/venv/bin/python scripts/split_captures.py --source data/raw/captures --output data/finetune
  finetune : 166 images, 103 labels  (captures_1.5/images:84, captures_1/images:82)
  test     : 41 images, 26 labels  (captures_1.5/images:21, captures_1/images:20)

Pool fine-tuning -> data/finetune/captures_finetune
Test compost     -> data/finetune/captures_test/data.yaml

Étapes suivantes (voir README) :
  python scripts/prepare_dataset.py --source data/finetune/captures_finetune --output data/finetune/dataset_finetune --ratios 0.85 0.15 0
  python scripts/evaluate.py --weights <pretrain.pt> --data data/finetune/captures_test/data.yaml --split test          # éval B
  python scripts/train.py --model <pretrain.pt> --data data/finetune/dataset_finetune/data.yaml --epochs 30 --lr0 0.001   # fine-tune
  python scripts/evaluate.py --weights runs/train_xxx/weights/best.pt --data data/finetune/captures_test/data.yaml --split test   # éval C


## 2. Éval B — le modèle pré-entraîné sur le test compost (AVANT fine-tuning)

In [3]:
run(f"python scripts/evaluate.py --weights {PRETRAIN} "
    f"--data data/finetune/captures_test/data.yaml --split test --device {DEVICE}")


$ /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/venv/bin/python scripts/evaluate.py --weights /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/runs/train_29-06_17h13/weights/best.pt --data data/finetune/captures_test/data.yaml --split test --device 0
Ultralytics 8.4.65 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 73 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4583.2±1230.6 MB/s, size: 251.3 KB)
val: Scanning /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/data/finetune/captures_test/labels/test... 26 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 41/41 751.1it/s 0.1s
val: New cache created: /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/data/finetune/captures_test/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.6s/it 4.9s2.3s
                   al

## 3. Fine-tuning — repart du pré-entraîné, learning rate bas

In [4]:
run(f"python scripts/train.py --model {PRETRAIN} "
    f"--data data/finetune/dataset_finetune/data.yaml "
    f"--epochs {EPOCHS} --lr0 0.001 --batch {BATCH} --device {DEVICE}")


$ /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/venv/bin/python scripts/train.py --model /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/runs/train_29-06_17h13/weights/best.pt --data data/finetune/dataset_finetune/data.yaml --epochs 50 --lr0 0.001 --batch 8 --device 0
Device : 0
New https://pypi.org/project/ultralytics/8.4.86 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.65 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/data/finetune/dataset_finetune/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.0it/s 1.0s2.2s
                   all         26         30      0.365      0.346      0.183      0.104

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       5/50      1.28G      1.335      2.492      1.284          8        640: 100% ━━━━━━━━━━━━ 18/18 1.1it/s 16.3s0.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.0it/s 1.0s2.2s
                   all         26         30      0.366      0.318      0.229      0.132

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       6/50      1.28G      1.261      2.197      1.238          6        640: 100% ━━━━━━━━━━━━ 18/18 1.3it/s 14.2s0.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.0it/s 1.0s2.2s
                   all         26  


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      24/50      1.28G      1.029      1.204      1.088          7        640: 100% ━━━━━━━━━━━━ 18/18 1.3it/s 13.6s0.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.1it/s 1.0s2.1s
                   all         26         30      0.824      0.701      0.736      0.549

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      25/50      1.28G      0.907      1.021      1.031         12        640: 100% ━━━━━━━━━━━━ 18/18 1.2it/s 14.5s0.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.1it/s 1.0s2.1s
                   all         26         30      0.885      0.682      0.739      0.526

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      26/50      1.28G      1.014      1.053      1.081          9        640: 100% ━


42 epochs completed in 0.183 hours.
Optimizer stripped from /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/runs/train_03-07_05h25/weights/last.pt, 6.2MB
Optimizer stripped from /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/runs/train_03-07_05h25/weights/best.pt, 6.2MB

Validating /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/runs/train_03-07_05h25/weights/best.pt...
Ultralytics 8.4.65 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 73 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 8.8it/s 0.2s
                   all         26         30      0.844       0.62      0.728      0.564
             Plastique          5          8       0.78      0.375      0.438      0.386
                 Metal          9         15      0.764      0.435      0.603      0.423
             Aluminium      

## 4. Éval C — le modèle fine-tuné sur le MÊME test compost (APRÈS)
Le dernier `runs/train_*` est trouvé automatiquement.

In [5]:
finetuned = max(glob.glob("runs/train_*/weights/best.pt"), key=os.path.getmtime)
print("modèle fine-tuné :", finetuned)
run(f"python scripts/evaluate.py --weights {finetuned} "
    f"--data data/finetune/captures_test/data.yaml --split test --device {DEVICE}")
print("\nTerminé. Compare les deux derniers dossiers runs/eval_* : B (avant) vs C (après).")

modèle fine-tuné : runs/train_03-07_05h25/weights/best.pt

$ /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/venv/bin/python scripts/evaluate.py --weights runs/train_03-07_05h25/weights/best.pt --data data/finetune/captures_test/data.yaml --split test --device 0
Ultralytics 8.4.65 🚀 Python-3.12.3 torch-2.12.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Model summary (fused): 73 layers, 3,007,013 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 132.9±69.8 MB/s, size: 275.6 KB)
val: Scanning /home/aliikched/stage/Compost_Waste_Yolo/compost-yolo/data/finetune/captures_test/labels/test.cache... 26 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 41/41 8.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.0it/s 3.0s2.3s
                   all         41         56      0.731      0.354      0.345      0.238
             Plastique          9          9      